In [ ]:
import tinytensor as tt
import numpy as np

#### Gradient vs. Jacobian

The **jacobian** holds information on how sensitive each output parameter is, in respect to variation of every input parameter. Therefore, the shape of the jacobian
has to be the _procuct_ of the output and input shape.

$$
\begin{align*}
    \text{operation}: \quad &(\texttt m,) \to (\texttt n,)\qquad&& \Leftrightarrow \qquad\mathbb K^m\to \mathbb K^n\mskip{5em}\\
    \Rightarrow\quad \text{jacobian}:\quad&  (\texttt n ,\texttt m)&& \Leftrightarrow \qquad~\in\mathbb K^{n\times m}
\end{align*}
$$

For a neural network that results in a scalar loss, this means that the *"partial"* jacobian from any parameter (tensor) to the loss has the **same shape** as this parameter (tensor). 

In this project, jacobians for functions returning a scalar (i.e., that have the same shape as its input) are called **gradients**. This distinction makes more sense when looking at the different ways a gradient can be calculated:

1. Using the chain rule, the gradient (or any jacobian) can be calculated as the matrixmultiplication of all the _intermediate_ jacobians.
$$
\begin{align*}
    \text{operation}: \quad &(\texttt m,) \to (\texttt n,) \to (,)\qquad&& \Leftrightarrow \qquad\mathbb K^m\to \mathbb K^n\to\mathbb K\mskip{5em}\\
    \Rightarrow\quad \text{jacobian}:\quad&  (\texttt{n},) \cdot (\texttt n,\texttt m) = (\texttt m,)&& \Leftrightarrow \qquad~ J_g\cdot J_f\in\mathbb K^m, J_g\in\mathbb K^n, J_f\in \mathbb K^{n\times m}
\end{align*}
$$
2. For chains of multilinear operations (like the ones supported by `einsum()`) that result in a scalar, this method uses more calculations and memory than necessary, especially since most of the entries in the intermdediate Jacobians are zeros or identical across an axis, which get contracted at some point. Instead, the gradient can be constructed in its final shape in a more direct way.


**example**

$W\in\mathbb K^{3\times 2}$, $x\in\mathbb K^2$:
$$
h = W x \quad (\texttt{"ij,j->i"}), \qquad L = h\cdot h \quad (\texttt{"i,i->"})
$$

Since $h$ appears twice in $L$, the contributions of both positions are summed (product rule).

In [ ]:
W = tt.Tensor(np.random.normal(size=(3, 2)), calculateGrad=True)
x = tt.Tensor([1.0, 2.0])

h = tt.einsum([W, x], "ij,j->i")
L = tt.einsum([h, h], "i,i->")

# 1. chain rule: contract the intermediate jacobians
J_L_h = tt.partial_jacobian([h, h], "i,i->", 0).data + tt.partial_jacobian([h, h], "i,i->", 1).data
J_h_W = tt.partial_jacobian([W, x], "ij,j->i", 0).data
grad_jacobian = np.einsum("a,abc->bc", J_L_h, J_h_W)
print(f"J_L_h: {J_L_h.shape}, J_h_W: {J_h_W.shape} ({J_h_W.size} entries, {np.count_nonzero(J_h_W)} non-zero)")

# 2. direct: pass the gradient backwards in its final shape
one = tt.Tensor(np.array(1.0))
grad_h = tt.partial_gradient([h, h], "i,i->", 0, one).data + tt.partial_gradient([h, h], "i,i->", 1, one).data
grad_direct = tt.partial_gradient([W, x], "ij,j->i", 0, tt.Tensor(grad_h)).data
print(f"grad_h: {grad_h.shape}, grad_W: {grad_direct.shape}")

# backward() uses method 2 automatically
L.grad = np.array(1.0)
L.backward()

print(f"\nchain rule:\n{grad_jacobian}\ndirect:\n{grad_direct}\nbackward():\n{W.grad}")
print(f"\nall equal: {np.allclose(grad_jacobian, grad_direct) and np.allclose(grad_direct, W.grad)}")

#### 2-Layer Neural Network

**linear layer class**

Weights initialized around `mean` $=0$ with `std` $=\sqrt{2/\texttt{inSize}}$.  
Bias initialized as $0$.

Forward pass as a matrix multiplication and addition:
`out = weights @ batch + bias`.

In [ ]:
class Linear:
    def __init__(self, inSize, outSize):
        self.weights = tt.Tensor(np.random.normal(0, np.sqrt(2 / inSize), (outSize, inSize)), calculateGrad=True)
        self.bias = tt.Tensor(np.zeros((outSize,1)), calculateGrad=True)

    # forward
    def __call__(self, x: tt.Tensor):
        return self.weights @ x + self.bias

    def update(self, step):
        self.weights.data -= self.weights.grad * step
        self.bias.data -= self.bias.grad * step

    def zeroGrad(self):
        self.weights.grad *= 0
        self.bias.grad *= 0

**architecture** & **initialization**

- `layer1`: (4, batch) -> (8, batch)  
- `relu`: (8, batch) -> (8, batch)  
- `layer2`: (8, batch) -> (2, batch)
- `squared error loss`: (2, batch) -> (,)

In [ ]:
layer1 = Linear(4, 8)
layer2 = Linear(8, 2)

# training data (batch of 2)
X = tt.Tensor(np.transpose([[0.1, 0.8, 0.8, 0.1], [0.9, 0.2, 0.8, 0.1]]))
y = tt.Tensor([[1, 0.5], [0.5, 1]])

**training loop**

In [ ]:
epochs = 500
step = 1e-2
for epoch in range(epochs):
    # forward pass
    X1 = tt.relu(layer1(X))
    diff = layer2(X1) - y
    loss = tt.einsum([diff, diff], "ij,ij->")

    # backward pass
    loss.grad = np.array(1)
    loss.backward()

    # optimization step
    for layer in (layer1, layer2):
        layer.update(step)
        layer.zeroGrad()

    if epoch % 50 == 0:
        print(f"epoch: {epoch:3}, loss: {loss.data:.2e}")

print(f"\nprediction after {epochs} epochs: \n{layer2(tt.relu(layer1(X))).data}")

#### Comparison with PyTorch

The same 2-layer network as above, with freshly initialized parameters. The parameters are copied into PyTorch tensors, and both libraries compute the gradients of the squared error loss with `.backward()`.

In [ ]:
import torch

layer1 = Linear(4, 8)
layer2 = Linear(8, 2)
tt_params = [layer1.weights, layer1.bias, layer2.weights, layer2.bias]

# tinytensor
diff = layer2(tt.relu(layer1(X))) - y
loss = tt.einsum([diff, diff], "ij,ij->")
loss.grad = np.array(1.0)
loss.backward()

# pytorch, with the same parameters and data
W1, b1, W2, b2 = [torch.tensor(p.data, requires_grad=True) for p in tt_params]
X_torch, y_torch = torch.tensor(X.data), torch.tensor(y.data)
loss_torch = ((W2 @ torch.relu(W1 @ X_torch + b1) + b2 - y_torch) ** 2).sum()
loss_torch.backward()

print(f"loss: tinytensor {loss.data:.6f}, pytorch {loss_torch.item():.6f}\n")
for name, p, p_torch in zip(["W1", "b1", "W2", "b2"], tt_params, [W1, b1, W2, b2]):
    print(f"{name}: max. difference {np.abs(p.grad - p_torch.grad.numpy()).max():.1e}")